# Import Library

In [1]:
import os, glob, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings('ignore')
PAL = ['#093C5D', '#3B7597', '#6FD1D7', '#5DF8D8']
INK = '#093C5D'
SPLIT_C = {'train': PAL[1], 'test': PAL[2]}
CMAP = LinearSegmentedColormap.from_list('gelar', PAL)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tif', '.tiff'}
SEED = 42

sns.set_theme(style='whitegrid', rc={
    'axes.edgecolor': INK, 'axes.labelcolor': INK, 'axes.titlecolor': INK, 'axes.titleweight': 'bold',
    'text.color': INK, 'xtick.color': INK, 'ytick.color': INK, 'grid.color': '#DCEBF0',
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110})


def show_grid(paths, titles=None, ncols=6, size=2.2, title=None):
    paths = list(paths); n = len(paths)
    titles = list(titles) if titles is not None else [None] * n
    nr = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nr, ncols, figsize=(ncols * size, nr * size * 1.1))
    axs = np.ravel(axs)
    for a in axs: a.axis('off')
    for a, p, t in zip(axs, paths, titles):
        try: a.imshow(Image.open(p).convert('RGB'))
        except Exception: a.text(.5, .5, 'unreadable', ha='center', va='center', transform=a.transAxes, color=INK)
        if t: a.set_title(str(t), fontsize=7)
    if title: fig.suptitle(title, fontweight='bold', color=INK)
    plt.tight_layout(rect=[0, 0, 1, .97]); plt.show()


def dist_plot(col, label, log=False, bins=40):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={'width_ratios': [2, 1]})
    sns.histplot(data=imgs, x=col, hue='split', palette=SPLIT_C, bins=bins, kde=True, element='step', log_scale=log, ax=ax[0])
    sns.boxplot(data=imgs, x='split', y=col, palette=SPLIT_C, ax=ax[1])
    if log: ax[1].set_yscale('log')
    ax[0].set_title(f'Distribusi {label}'); ax[0].set_xlabel(label)
    ax[1].set_title('Boxplot'); ax[1].set_xlabel(''); ax[1].set_ylabel(label)
    plt.tight_layout(); plt.show()


def count_plot(col, title):
    d = imgs.assign(**{col: imgs[col].astype(str)})
    fig, ax = plt.subplots(figsize=(7, 3.6))
    sns.countplot(data=d, x=col, hue='split', palette=SPLIT_C, order=d[col].value_counts().index, ax=ax)
    for c in ax.containers: ax.bar_label(c, padding=2, fontsize=8)
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel('Jumlah')
    plt.tight_layout(); plt.show()